# EDA — Klasyfikacja Guzów Mózgu MRI

**Dataset:** Brain Tumor MRI Dataset (Masoud Nickparvar, Kaggle)  
**Klasy:** glioma, meningioma, notumor, pituitary  
**Łączna liczba obrazów:** ~7023  

Ten notebook wykonuje eksploracyjną analizę danych przed trenowaniem modelu EfficientNetB0.

In [1]:
import os
import numpy as np
import matplotlib
matplotlib.use('Agg')  # bez GUI — bezpieczne na serwerze/SSH
import matplotlib.pyplot as plt
from pathlib import Path
from collections import Counter
from PIL import Image
import warnings
warnings.filterwarnings('ignore')

# Ścieżki
BASE_DIR = Path('..').resolve()
DATA_DIR = BASE_DIR / 'data' / 'raw'
TRAIN_DIR = DATA_DIR / 'Training'
TEST_DIR  = DATA_DIR / 'Testing'
OUT_DIR   = BASE_DIR / 'out'
OUT_DIR.mkdir(exist_ok=True)

CLASSES = ['glioma', 'meningioma', 'notumor', 'pituitary']
COLORS  = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12']

DATA_AVAILABLE = TRAIN_DIR.exists() and any(TRAIN_DIR.iterdir())
print(f'Katalog danych: {DATA_DIR}')
print(f'Dane dostępne:  {DATA_AVAILABLE}')
if not DATA_AVAILABLE:
    print('\n[!] Brak danych — uruchom komórki aby zobaczyć strukturę kodu.')
    print('    Pobierz dataset: kaggle datasets download -d masoudnickparvar/brain-tumor-mri-dataset --unzip')
    print('    Wypakuj do: data/raw/')

Katalog danych: /home/balicz3k/Documents/master-studies/Medical/BrainTumorClassification/data/raw
Dane dostępne:  True


## 1. Rozkład klas

Sprawdzamy ile obrazów należy do każdej klasy w zbiorze treningowym i testowym.

In [2]:
def count_images(split_dir):
    """Zlicza obrazy per klasa w podanym katalogu."""
    counts = {}
    if not split_dir.exists():
        return counts
    for cls in CLASSES:
        cls_dir = split_dir / cls
        if cls_dir.exists():
            imgs = [f for f in cls_dir.iterdir() if f.suffix.lower() in ('.jpg', '.jpeg', '.png')]
            counts[cls] = len(imgs)
        else:
            counts[cls] = 0
    return counts

train_counts = count_images(TRAIN_DIR)
test_counts  = count_images(TEST_DIR)

if DATA_AVAILABLE:
    print('=== Rozkład klas ===')
    print(f'{'Klasa':<15} {'Train':>8} {'Test':>8} {'Łącznie':>10}')
    print('-' * 45)
    total_train, total_test = 0, 0
    for cls in CLASSES:
        tr = train_counts.get(cls, 0)
        te = test_counts.get(cls, 0)
        total_train += tr
        total_test  += te
        print(f'{cls:<15} {tr:>8} {te:>8} {tr+te:>10}')
    print('-' * 45)
    print(f'{'ŁĄCZNIE':<15} {total_train:>8} {total_test:>8} {total_train+total_test:>10}')
else:
    # Dane referencyjne z dokumentacji datasetu
    train_counts = {'glioma': 1321, 'meningioma': 1339, 'notumor': 1595, 'pituitary': 1457}
    test_counts  = {'glioma':  300, 'meningioma':  306, 'notumor':  405, 'pituitary':  300}
    print('[!] Używam danych referencyjnych z dokumentacji datasetu (brak lokalnych danych)')
    print(f'{'Klasa':<15} {'Train':>8} {'Test':>8} {'Łącznie':>10}')
    print('-' * 45)
    for cls in CLASSES:
        tr = train_counts[cls]
        te = test_counts[cls]
        print(f'{cls:<15} {tr:>8} {te:>8} {tr+te:>10}')

=== Rozkład klas ===
Klasa              Train     Test    Łącznie
---------------------------------------------
glioma              1400      400       1800
meningioma          1400      400       1800
notumor             1400      400       1800
pituitary           1400      400       1800
---------------------------------------------
ŁĄCZNIE             5600     1600       7200


In [3]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (counts, title) in zip(axes, [(train_counts, 'Trening'), (test_counts, 'Test')]):
    bars = ax.bar(CLASSES, [counts.get(c, 0) for c in CLASSES], color=COLORS, edgecolor='black', linewidth=0.8)
    ax.set_title(f'Rozkład klas — {title}', fontsize=13, fontweight='bold')
    ax.set_xlabel('Klasa', fontsize=11)
    ax.set_ylabel('Liczba obrazów', fontsize=11)
    ax.set_ylim(0, max([counts.get(c, 0) for c in CLASSES]) * 1.15)
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h + 15, str(int(h)),
                ha='center', va='bottom', fontsize=10, fontweight='bold')
    ax.tick_params(axis='x', rotation=15)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.tight_layout()
out_path = OUT_DIR / 'class_distribution.png'
plt.savefig(out_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Wykres zapisany: {out_path}')

Wykres zapisany: /home/balicz3k/Documents/master-studies/Medical/BrainTumorClassification/out/class_distribution.png


## 2. Przykładowe obrazy MRI

Siatka 3×4: 3 losowe obrazy z każdej z 4 klas.

In [4]:
import random

if DATA_AVAILABLE:
    fig, axes = plt.subplots(3, 4, figsize=(14, 10))
    fig.suptitle('Przykładowe obrazy MRI — zbiór treningowy', fontsize=14, fontweight='bold', y=1.01)

    random.seed(42)
    for col_idx, cls in enumerate(CLASSES):
        cls_dir = TRAIN_DIR / cls
        all_imgs = [f for f in cls_dir.iterdir() if f.suffix.lower() in ('.jpg', '.jpeg', '.png')]
        sampled  = random.sample(all_imgs, min(3, len(all_imgs)))

        for row_idx, img_path in enumerate(sampled):
            ax = axes[row_idx][col_idx]
            img = Image.open(img_path).convert('L')  # grayscale
            ax.imshow(img, cmap='gray')
            ax.set_title(cls.upper() if row_idx == 0 else '', fontsize=11,
                         fontweight='bold', color=COLORS[col_idx])
            ax.axis('off')

    plt.tight_layout()
    out_path = OUT_DIR / 'sample_images.png'
    plt.savefig(out_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Wykres zapisany: {out_path}')
else:
    print('[!] Brak danych lokalnych — załaduj dataset aby wyświetlić przykładowe obrazy MRI.')
    print('    Po pobraniu danych ta komórka wyświetli siatkę 3x4 (3 obrazy x 4 klasy).')

Wykres zapisany: /home/balicz3k/Documents/master-studies/Medical/BrainTumorClassification/out/sample_images.png


## 3. Statystyki wymiarów obrazów

Sprawdzamy rozdzielczość (szerokość × wysokość) wszystkich obrazów — min, max, średnia.

In [5]:
if DATA_AVAILABLE:
    dim_stats = {cls: {'widths': [], 'heights': []} for cls in CLASSES}

    for cls in CLASSES:
        cls_dir = TRAIN_DIR / cls
        all_imgs = [f for f in cls_dir.iterdir() if f.suffix.lower() in ('.jpg', '.jpeg', '.png')]
        # Próbkujemy max 200 obrazów dla szybkości
        sample = random.sample(all_imgs, min(200, len(all_imgs)))
        for img_path in sample:
            with Image.open(img_path) as img:
                w, h = img.size
                dim_stats[cls]['widths'].append(w)
                dim_stats[cls]['heights'].append(h)

    print('=== Statystyki wymiarów (próbka 200 obrazów/klasę, zbiór treningowy) ===')
    print(f'{'Klasa':<15} {'Szer. min':>10} {'Szer. max':>10} {'Szer. mean':>12} {'Wys. mean':>10}')
    print('-' * 62)
    all_widths, all_heights = [], []
    for cls in CLASSES:
        ws = dim_stats[cls]['widths']
        hs = dim_stats[cls]['heights']
        all_widths.extend(ws)
        all_heights.extend(hs)
        print(f'{cls:<15} {min(ws):>10} {max(ws):>10} {np.mean(ws):>12.1f} {np.mean(hs):>10.1f}')
    print('-' * 62)
    print(f'{'ŁĄCZNIE':<15} {min(all_widths):>10} {max(all_widths):>10} {np.mean(all_widths):>12.1f} {np.mean(all_heights):>10.1f}')
    print('\n[i] Obrazy będą przeskalowane do 224x224 przed podaniem do modelu.')
else:
    print('[!] Brak danych lokalnych.')
    print('    Obrazy w datasecie mają zróżnicowane wymiary (głównie ok. 512x512).')
    print('    Przed trenowaniem zostaną przeskalowane do 224x224 (wymaganie EfficientNetB0).')

=== Statystyki wymiarów (próbka 200 obrazów/klasę, zbiór treningowy) ===
Klasa            Szer. min  Szer. max   Szer. mean  Wys. mean
--------------------------------------------------------------
glioma                 512        512        512.0      512.0
meningioma             200        512        501.3      503.0
notumor                150       1000        309.2      324.9
pituitary              225       1168        508.7      508.7
--------------------------------------------------------------
ŁĄCZNIE                150       1168        457.8      462.1

[i] Obrazy będą przeskalowane do 224x224 przed podaniem do modelu.


## 4. Statystyki pikseli — intensywność per klasa

Średnia i odchylenie standardowe wartości pikseli dla każdej klasy (skala 0–255).

In [6]:
if DATA_AVAILABLE:
    pixel_stats = {}

    for cls in CLASSES:
        cls_dir = TRAIN_DIR / cls
        all_imgs = [f for f in cls_dir.iterdir() if f.suffix.lower() in ('.jpg', '.jpeg', '.png')]
        sample = random.sample(all_imgs, min(100, len(all_imgs)))
        means, stds = [], []
        for img_path in sample:
            arr = np.array(Image.open(img_path).convert('L'), dtype=np.float32)
            means.append(arr.mean())
            stds.append(arr.std())
        pixel_stats[cls] = {'mean': np.mean(means), 'std': np.mean(stds)}

    print('=== Statystyki intensywności pikseli (grayscale, próbka 100/klasę) ===')
    print(f'{'Klasa':<15} {'Średnia':>10} {'Std dev':>10}')
    print('-' * 38)
    for cls in CLASSES:
        m = pixel_stats[cls]['mean']
        s = pixel_stats[cls]['std']
        print(f'{cls:<15} {m:>10.2f} {s:>10.2f}')

    # Wykres słupkowy
    fig, ax = plt.subplots(figsize=(9, 5))
    x = np.arange(len(CLASSES))
    means_vals = [pixel_stats[c]['mean'] for c in CLASSES]
    stds_vals  = [pixel_stats[c]['std']  for c in CLASSES]
    bars = ax.bar(x, means_vals, yerr=stds_vals, color=COLORS, edgecolor='black',
                  linewidth=0.8, capsize=5, alpha=0.85)
    ax.set_xticks(x)
    ax.set_xticklabels(CLASSES, fontsize=11)
    ax.set_title('Średnia intensywność pikseli per klasa (± std)', fontsize=13, fontweight='bold')
    ax.set_ylabel('Intensywność (0–255)', fontsize=11)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    plt.tight_layout()
    out_path = OUT_DIR / 'pixel_intensity.png'
    plt.savefig(out_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Wykres zapisany: {out_path}')
else:
    print('[!] Brak danych lokalnych — statystyki pikseli zostaną obliczone po pobraniu danych.')

=== Statystyki intensywności pikseli (grayscale, próbka 100/klasę) ===
Klasa              Średnia    Std dev
--------------------------------------
glioma               34.51      38.93
meningioma           41.90      46.14
notumor              64.26      62.14
pituitary            50.62      41.71
Wykres zapisany: /home/balicz3k/Documents/master-studies/Medical/BrainTumorClassification/out/pixel_intensity.png


## 5. Plan podziału danych (Train/Val/Test)

Planowany stratyfikowany podział: **70% trening / 15% walidacja / 15% test**.

In [7]:
from sklearn.model_selection import train_test_split

# Dane referencyjne (z dokumentacji lub zliczone wyżej)
ref_counts = {
    'glioma':     train_counts.get('glioma',     1321),
    'meningioma': train_counts.get('meningioma', 1339),
    'notumor':    train_counts.get('notumor',    1595),
    'pituitary':  train_counts.get('pituitary',  1457),
}

print('=== Plan podziału stratyfikowanego (70 / 15 / 15) ===')
print(f'{'Klasa':<15} {'Łącznie':>9} {'Train (70%)':>12} {'Val (15%)':>11} {'Test (15%)':>11}')
print('-' * 62)
grand_total = grand_train = grand_val = grand_test = 0
for cls, total in ref_counts.items():
    n_train = int(total * 0.70)
    n_val   = int(total * 0.15)
    n_test  = total - n_train - n_val
    grand_total += total
    grand_train += n_train
    grand_val   += n_val
    grand_test  += n_test
    print(f'{cls:<15} {total:>9} {n_train:>12} {n_val:>11} {n_test:>11}')
print('-' * 62)
print(f'{'ŁĄCZNIE':<15} {grand_total:>9} {grand_train:>12} {grand_val:>11} {grand_test:>11}')
print(f'{'UDZIAŁ':<15} {'100%':>9} {grand_train/grand_total*100:>11.1f}% {grand_val/grand_total*100:>10.1f}% {grand_test/grand_total*100:>10.1f}%')

print('\n=== Uwagi do stratyfikacji ===')
print('- Podział zachowuje proporcje klas w każdym splicie')
print('- Klasa meningioma jest najmniej liczna — ważne dla stratyfikacji')
print('- Użyjemy sklearn.model_selection.train_test_split z stratify=labels')
print('- Dane testowe NIE są używane do walidacji hiperparametrów')

=== Plan podziału stratyfikowanego (70 / 15 / 15) ===
Klasa             Łącznie  Train (70%)   Val (15%)  Test (15%)
--------------------------------------------------------------
glioma               1400          979         210         211
meningioma           1400          979         210         211
notumor              1400          979         210         211
pituitary            1400          979         210         211
--------------------------------------------------------------
ŁĄCZNIE              5600         3916         840         844
UDZIAŁ               100%        69.9%       15.0%       15.1%

=== Uwagi do stratyfikacji ===
- Podział zachowuje proporcje klas w każdym splicie
- Klasa meningioma jest najmniej liczna — ważne dla stratyfikacji
- Użyjemy sklearn.model_selection.train_test_split z stratify=labels
- Dane testowe NIE są używane do walidacji hiperparametrów


## 6. Sprawdzenie środowiska

Weryfikacja PyTorch i dostępności GPU (RTX 4060).

In [8]:
import torch
import torchvision

print('=== Środowisko ===')
print(f'PyTorch:     {torch.__version__}')
print(f'Torchvision: {torchvision.__version__}')
print(f'CUDA dostępne: {torch.cuda.is_available()}')

if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    mem_total = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f'VRAM: {mem_total:.1f} GB')
    print('[OK] GPU gotowy do trenowania!')
else:
    print('[!] GPU niedostępne — trening będzie wolniejszy na CPU')

print('\n=== Podsumowanie EDA ===')
print(f'Klasy:              {len(CLASSES)} ({', '.join(CLASSES)})')
total_all = sum(train_counts.values()) + sum(test_counts.values())
print(f'Łączna liczba obrazów (train+test): {total_all}')
print(f'Nierównowaga klas:  {'TAK — sprawdź class weights' if max(train_counts.values()) / min(train_counts.values()) > 1.3 else 'Niewielka'}')
print(f'Docelowy rozmiar wejścia modelu: 224x224 px')
print(f'Model bazowy: EfficientNetB0 (pretrained ImageNet)')
print(f'Planowany podział: 70/15/15 stratifikowany')

=== Środowisko ===
PyTorch:     2.15.0+cpu
Torchvision: 0.29.2+cpu
CUDA dostępne: False
[!] GPU niedostępne — trening będzie wolniejszy na CPU

=== Podsumowanie EDA ===
Klasy:              4 (glioma, meningioma, notumor, pituitary)
Łączna liczba obrazów (train+test): 7200
Nierównowaga klas:  Niewielka
Docelowy rozmiar wejścia modelu: 224x224 px
Model bazowy: EfficientNetB0 (pretrained ImageNet)
Planowany podział: 70/15/15 stratifikowany
